# Credit Risk Scoring — Logistic Regression + XGBoost + Gini/KS

**Auteur :** Emmanuel TSAGUE — EDF MAD EDVANCE | Datascientest 2024  
**Données :** simulées — 10 000 demandes de crédit

### Plan
1. Exploration — taux de défaut
2. Analyse du risque par variable
3. Comparaison de modèles (AUC)
4. Courbe ROC + KS statistic
5. Distribution des scores
6. Métriques Gini + KS

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.data_generation import load_or_generate
from src.credit_risk import (build_pipeline, compare_models,
                               evaluate_credit_model, gini_coefficient,
                               ks_statistic, plot_score_distribution)
print('Imports OK')

## 1. Exploration

In [ ]:
df = load_or_generate('../data_sample/credit_risk_simulated.csv')
print(f'Clients : {len(df):,} | Taux défaut : {df.defaut.mean():.1%}')
df.head()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
cols = ['age','revenu_annuel','montant_credit','duree_mois',
        'anciennete_emploi','nb_credits_en_cours','taux_endettement']
for ax, col in zip(axes.flat, cols):
    df[df.defaut==0][col].hist(ax=ax, alpha=0.6, bins=30, label='Solvable', color='#4CAF50')
    df[df.defaut==1][col].hist(ax=ax, alpha=0.7, bins=30, label='Défaut',   color='#F44336')
    ax.set_title(col); ax.legend(fontsize=7)
plt.suptitle('Profils : solvables vs défaut', fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Taux de défaut par historique et type de crédit

In [ ]:
print('Taux de défaut par historique :')
print(df.groupby('historique')['defaut'].mean().round(3))
print('\nTaux de défaut par type de crédit :')
print(df.groupby('type_credit')['defaut'].mean().round(3))

## 3. Comparaison de modèles

In [ ]:
from src.credit_risk import NUM_FEATURES, CAT_FEATURES

X = df[NUM_FEATURES + CAT_FEATURES]
y = df['defaut']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Comparaison AUC — cross-validation 5-fold :')
cv_res = compare_models(X_train, y_train)

## 4. Évaluation — ROC + KS

In [ ]:
pipe = build_pipeline('xgb')
pipe.fit(X_train, y_train)
result = evaluate_credit_model(pipe, X_test, y_test)

## 5. Distribution des scores

In [ ]:
plot_score_distribution(y_test.values, result['y_proba'])

## 6. Métriques réglementaires (Basel III)

In [ ]:
gini = gini_coefficient(y_test, result['y_proba'])
ks   = ks_statistic(y_test, result['y_proba'])
print(f'Coefficient Gini = {gini:.4f}')
print(f'KS Statistic     = {ks:.4f}')
print(f'ROC-AUC          = {result["auc"]:.4f}')

## Synthèse

| Métrique | Valeur |
|----------|--------|
| AUC | ~0.87 |
| Gini | ~0.74 |
| KS | ~0.62 |

> **Top variables :** historique, taux_endettement, nb_credits_en_cours  

*Données simulées (seed=42) — Emmanuel TSAGUE*